In [1]:
!pip install -q datasets pandas numpy tqdm

In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata
import os
import json
from tqdm.auto import tqdm

from datasets import load_dataset

print("Libraries loaded successfully")

Libraries loaded successfully


In [3]:
# Target FINAL cleaned dataset size
TARGET_FINAL_MB = 150

# We collect more raw data because cleaning will reduce the size
TARGET_RAW_MB = 200

TARGET_RAW_BYTES = TARGET_RAW_MB * 1024 * 1024
TARGET_FINAL_BYTES = TARGET_FINAL_MB * 1024 * 1024

print("Target raw size:", TARGET_RAW_MB, "MB")
print("Target final cleaned size:", TARGET_FINAL_MB, "MB")

Target raw size: 200 MB
Target final cleaned size: 150 MB


In [4]:
def normalize_unicode(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    return text

In [5]:
def remove_urls(text):
    url_pattern = r'https?://\S+|www\.\S+'
    return re.sub(url_pattern, ' ', text)

In [6]:
def remove_control_characters(text):
    cleaned = []

    for char in text:
        category = unicodedata.category(char)

        # Keep normal characters, spaces, and newlines
        if category.startswith("C"):
            if char in "\n\t":
                cleaned.append(char)
        else:
            cleaned.append(char)

    return "".join(cleaned)

In [7]:
def normalize_whitespace(text):
    # Normalize spaces/tabs
    text = re.sub(r'[ \t]+', ' ', text)

    # Remove excessive blank lines
    text = re.sub(r'\n\s*\n+', '\n\n', text)

    return text.strip()

In [8]:
def clean_text(text):
    text = normalize_unicode(text)
    text = remove_urls(text)
    text = remove_control_characters(text)
    text = normalize_whitespace(text)

    return text

In [10]:
hindi_stream = load_dataset(
    "ai4bharat/IndicCorpV2",
    "indiccorp_v2",
    split="hin_Deva",
    streaming=True
)

print("Hindi streaming dataset loaded")

Hindi streaming dataset loaded


In [11]:
for i, row in enumerate(hindi_stream):
    print("Record", i)
    print(row)
    print("-" * 80)

    if i >= 4:
        break

Record 0
{'text': 'लोगों को बिलों संबंधी सुविधा देना ही उनका काम'}
--------------------------------------------------------------------------------
Record 1
{'text': ''}
--------------------------------------------------------------------------------
Record 2
{'text': 'इनेलो 1987 में उस वक्त ऐसे ही दोराहे पर खड़ी थी, जब पूर्व उपप्रधानमंत्री देवीलाल ने अपने पुत्र ओमप्रकाश चौटाला को अपना राजनीतिक उत्तराधिकारी घोषित किया था। हालांकि तब पार्टी पर देवीलाल की मजबूत पकड़ के चलते पार्टी टूटने से बच गई थी। 1989 में देवीलाल केन्द्र की राजनीति में सक्रिय हो गए थे और उनके उपप्रधानमंत्री बनने के पश्चात् उनके तीन बेटों जगदीश सिंह, रणजीत सिंह और ओमप्रकाश चौटाला में से रणजीत और ओमप्रकाश के बीच हरियाणा में उनकी राजनीतिक विरासत को लेकर जंग शुरू हो गई थी। उन परिस्थितियों में देवीलाल ने कड़ा निर्णय लेते हुए पार्टी की बागडोर ओमप्रकाश चौटाला के हवाले कर दी थी, जिसके बाद रणजीत की बगावत का असर पार्टी, संगठन और उनकी सरकार पर भी पड़ा था। उस समय रणजीत की नाराजगी के चलते उनके समर्थन में कई कैबिनेट मंत्रियों ने इस

In [12]:
hindi_records = []

total_bytes = 0
row_count = 0

print("Collecting Hindi data...")

for row in tqdm(hindi_stream):

    text = row.get("text", "")

    if not isinstance(text, str):
        continue

    text = text.strip()

    if not text:
        continue

    size = len(text.encode("utf-8"))

    hindi_records.append(text)

    total_bytes += size
    row_count += 1

    if total_bytes >= TARGET_RAW_BYTES:
        break

print()
print("Hindi rows collected:", row_count)
print("Hindi raw size:", round(total_bytes / (1024 * 1024), 2), "MB")

0it [00:00, ?it/s]


Hindi rows collected: 278137
Hindi raw size: 200.0 MB


In [13]:
hindi_df = pd.DataFrame({
    "text": hindi_records
})

print("Hindi DataFrame shape:", hindi_df.shape)

hindi_df.head()

Hindi DataFrame shape: (278137, 1)


,text
0,लोगों को बिलों संबंधी सुविधा देना ही उनका काम
1,इनेलो 1987 में उस वक्त ऐसे ही दोराहे पर खड़ी थ...
2,जहां आई थी तबाही उस घाटी क्षेत्र में खतरा ज्यादा
3,इसके बाद केंद्र की ओर से प्रदेश सरकार को पीएमज...
4,यह पूछने पर कि इस बड़े मैच से पहले उनकी नींद ग...


In [14]:
hindi_df["clean_text"] = hindi_df["text"].apply(clean_text)

print("Hindi cleaning completed")

Hindi cleaning completed


In [15]:
hindi_df["clean_text"] = hindi_df["clean_text"].astype(str)

hindi_df = hindi_df[
    hindi_df["clean_text"].str.strip().str.len() > 0
].copy()

print("Rows after empty-text removal:", len(hindi_df))

Rows after empty-text removal: 278137


In [16]:
MIN_CHARS = 50

hindi_df = hindi_df[
    hindi_df["clean_text"].str.len() >= MIN_CHARS
].copy()

print("Rows after minimum-length filtering:", len(hindi_df))

Rows after minimum-length filtering: 238717


In [17]:
before = len(hindi_df)

hindi_df = hindi_df.drop_duplicates(
    subset=["clean_text"]
).reset_index(drop=True)

after = len(hindi_df)

print("Before duplicates:", before)
print("After duplicates:", after)
print("Duplicates removed:", before - after)

Before duplicates: 238717
After duplicates: 238698
Duplicates removed: 19


In [18]:
def devanagari_ratio(text):
    if not text:
        return 0

    devanagari_count = sum(
        1 for char in text
        if '\u0900' <= char <= '\u097F'
    )

    return devanagari_count / len(text)

In [19]:
hindi_df["devanagari_ratio"] = hindi_df["clean_text"].apply(
    devanagari_ratio
)

print(hindi_df["devanagari_ratio"].describe())

count    238698.000000
mean          0.783850
std           0.035534
min           0.227273
25%           0.771689
50%           0.789474
75%           0.803922
max           1.000000
Name: devanagari_ratio, dtype: float64


In [20]:
hindi_df = hindi_df[
    hindi_df["devanagari_ratio"] >= 0.30
].copy()

hindi_df = hindi_df.reset_index(drop=True)

print("Hindi rows after language filtering:", len(hindi_df))

Hindi rows after language filtering: 238693


In [21]:
hindi_df["bytes"] = hindi_df["clean_text"].apply(
    lambda x: len(x.encode("utf-8"))
)

hindi_size_mb = hindi_df["bytes"].sum() / (1024 * 1024)

print("Hindi cleaned size:", round(hindi_size_mb, 2), "MB")
print("Hindi records:", len(hindi_df))

Hindi cleaned size: 196.74 MB
Hindi records: 238693


In [22]:
hindi_selected = []
current_size = 0

for text in hindi_df["clean_text"]:

    size = len(text.encode("utf-8"))

    if current_size + size > TARGET_FINAL_BYTES:
        break

    hindi_selected.append(text)
    current_size += size

hindi_final = pd.DataFrame({
    "text": hindi_selected
})

print("Final Hindi records:", len(hindi_final))
print(
    "Final Hindi size:",
    round(current_size / (1024 * 1024), 2),
    "MB"
)

Final Hindi records: 182476
Final Hindi size: 150.0 MB


In [23]:
hindi_chars = hindi_final["text"].str.len()

print("Hindi Statistics")
print("=" * 50)

print("Total records:", len(hindi_final))
print("Total characters:", hindi_chars.sum())
print("Average characters:", round(hindi_chars.mean(), 2))
print("Minimum characters:", hindi_chars.min())
print("Maximum characters:", hindi_chars.max())
print("Median characters:", hindi_chars.median())
print("Size MB:", round(current_size / (1024 * 1024), 2))

Hindi Statistics
Total records: 182476
Total characters: 61369652
Average characters: 336.32
Minimum characters: 50
Maximum characters: 100259
Median characters: 254.0
Size MB: 150.0


In [26]:
hindi_stream = load_dataset(
    "ai4bharat/IndicCorpV2",
    "indiccorp_v2",
    split="hin_Deva",
    streaming=True
)

print("Hindi streaming dataset loaded")

Hindi streaming dataset loaded


In [27]:
for i, row in enumerate(hindi_stream):
    print(f"Record {i}")
    print(row)
    print("-" * 80)

    if i >= 4:
        break

Record 0
{'text': 'लोगों को बिलों संबंधी सुविधा देना ही उनका काम'}
--------------------------------------------------------------------------------
Record 1
{'text': ''}
--------------------------------------------------------------------------------
Record 2
{'text': 'इनेलो 1987 में उस वक्त ऐसे ही दोराहे पर खड़ी थी, जब पूर्व उपप्रधानमंत्री देवीलाल ने अपने पुत्र ओमप्रकाश चौटाला को अपना राजनीतिक उत्तराधिकारी घोषित किया था। हालांकि तब पार्टी पर देवीलाल की मजबूत पकड़ के चलते पार्टी टूटने से बच गई थी। 1989 में देवीलाल केन्द्र की राजनीति में सक्रिय हो गए थे और उनके उपप्रधानमंत्री बनने के पश्चात् उनके तीन बेटों जगदीश सिंह, रणजीत सिंह और ओमप्रकाश चौटाला में से रणजीत और ओमप्रकाश के बीच हरियाणा में उनकी राजनीतिक विरासत को लेकर जंग शुरू हो गई थी। उन परिस्थितियों में देवीलाल ने कड़ा निर्णय लेते हुए पार्टी की बागडोर ओमप्रकाश चौटाला के हवाले कर दी थी, जिसके बाद रणजीत की बगावत का असर पार्टी, संगठन और उनकी सरकार पर भी पड़ा था। उस समय रणजीत की नाराजगी के चलते उनके समर्थन में कई कैबिनेट मंत्रियों ने इस

In [28]:
# ============================================================
# 🇮🇳 HINDI — COLLECTION → CLEANING → QUALITY CHECK → EXPORT
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata
import os
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

TARGET_RAW_MB = 200
TARGET_FINAL_MB = 150

TARGET_RAW_BYTES = TARGET_RAW_MB * 1024 * 1024
TARGET_FINAL_BYTES = TARGET_FINAL_MB * 1024 * 1024

MIN_CHARS = 50

print("Hindi Bharat dataset preparation started")
print(f"Raw collection target   : {TARGET_RAW_MB} MB")
print(f"Final dataset target    : {TARGET_FINAL_MB} MB")
print(f"Minimum characters      : {MIN_CHARS}")


# ------------------------------------------------------------
# 2. Text cleaning functions
# ------------------------------------------------------------

def normalize_unicode(text):
    if not isinstance(text, str):
        return ""

    return unicodedata.normalize("NFKC", text)


def remove_urls(text):
    url_pattern = r'https?://\S+|www\.\S+'
    return re.sub(url_pattern, ' ', text)


def remove_control_characters(text):
    cleaned = []

    for char in text:
        category = unicodedata.category(char)

        if category.startswith("C"):
            if char in "\n\t":
                cleaned.append(char)
        else:
            cleaned.append(char)

    return "".join(cleaned)


def normalize_whitespace(text):
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n\n', text)

    return text.strip()


def clean_text(text):
    text = normalize_unicode(text)
    text = remove_urls(text)
    text = remove_control_characters(text)
    text = normalize_whitespace(text)

    return text


# ------------------------------------------------------------
# 3. Collect approximately 200 MB raw Hindi data
# ------------------------------------------------------------

hindi_records = []

total_raw_bytes = 0
rows_seen = 0
rows_collected = 0

print("\nCollecting Hindi data...")

for row in tqdm(hindi_stream):

    rows_seen += 1

    text = row.get("text", "")

    if not isinstance(text, str):
        continue

    text = text.strip()

    if not text:
        continue

    size = len(text.encode("utf-8"))

    hindi_records.append(text)

    total_raw_bytes += size
    rows_collected += 1

    if total_raw_bytes >= TARGET_RAW_BYTES:
        break


print("\nCollection complete")
print("-" * 60)
print("Rows seen:", rows_seen)
print("Rows collected:", rows_collected)
print(
    "Raw size:",
    round(total_raw_bytes / (1024 * 1024), 2),
    "MB"
)


# ------------------------------------------------------------
# 4. Create DataFrame
# ------------------------------------------------------------

hindi_df = pd.DataFrame({
    "text": hindi_records
})

print("\nInitial DataFrame:")
print("Rows:", len(hindi_df))
print("Columns:", list(hindi_df.columns))


# ------------------------------------------------------------
# 5. Clean text
# ------------------------------------------------------------

print("\nCleaning Hindi text...")

hindi_df["clean_text"] = hindi_df["text"].apply(clean_text)

print("Cleaning completed")


# ------------------------------------------------------------
# 6. Remove empty records
# ------------------------------------------------------------

before_empty = len(hindi_df)

hindi_df = hindi_df[
    hindi_df["clean_text"].str.strip().str.len() > 0
].copy()

after_empty = len(hindi_df)

print("\nEmpty-text removal")
print("Before:", before_empty)
print("After :", after_empty)
print("Removed:", before_empty - after_empty)


# ------------------------------------------------------------
# 7. Minimum-length filtering
# ------------------------------------------------------------

before_length = len(hindi_df)

hindi_df = hindi_df[
    hindi_df["clean_text"].str.len() >= MIN_CHARS
].copy()

after_length = len(hindi_df)

print("\nMinimum-length filtering")
print("Before:", before_length)
print("After :", after_length)
print("Removed:", before_length - after_length)


# ------------------------------------------------------------
# 8. Duplicate removal
# ------------------------------------------------------------

before_duplicates = len(hindi_df)

hindi_df = hindi_df.drop_duplicates(
    subset=["clean_text"]
).reset_index(drop=True)

after_duplicates = len(hindi_df)

print("\nDuplicate removal")
print("Before:", before_duplicates)
print("After :", after_duplicates)
print("Removed:", before_duplicates - after_duplicates)


# ------------------------------------------------------------
# 9. Hindi / Devanagari quality check
# ------------------------------------------------------------

def devanagari_ratio(text):
    if not text:
        return 0.0

    devanagari_count = sum(
        1
        for char in text
        if '\u0900' <= char <= '\u097F'
    )

    return devanagari_count / len(text)


hindi_df["devanagari_ratio"] = hindi_df["clean_text"].apply(
    devanagari_ratio
)

print("\nDevanagari ratio statistics")
print(hindi_df["devanagari_ratio"].describe())


# ------------------------------------------------------------
# 10. Keep Hindi-dominant records
# ------------------------------------------------------------

before_language = len(hindi_df)

hindi_df = hindi_df[
    hindi_df["devanagari_ratio"] >= 0.30
].copy()

hindi_df = hindi_df.reset_index(drop=True)

after_language = len(hindi_df)

print("\nHindi language filtering")
print("Before:", before_language)
print("After :", after_language)
print("Removed:", before_language - after_language)


# ------------------------------------------------------------
# 11. Calculate cleaned dataset size
# ------------------------------------------------------------

hindi_df["bytes"] = hindi_df["clean_text"].apply(
    lambda x: len(x.encode("utf-8"))
)

cleaned_size_bytes = hindi_df["bytes"].sum()

cleaned_size_mb = cleaned_size_bytes / (1024 * 1024)

print("\nCleaned Hindi dataset")
print("-" * 60)
print("Records:", len(hindi_df))
print("Size:", round(cleaned_size_mb, 2), "MB")


# ------------------------------------------------------------
# 12. Select approximately 150 MB final dataset
# ------------------------------------------------------------

hindi_selected = []
current_size = 0

for text in hindi_df["clean_text"]:

    size = len(text.encode("utf-8"))

    if current_size + size > TARGET_FINAL_BYTES:
        break

    hindi_selected.append(text)
    current_size += size


hindi_final = pd.DataFrame({
    "text": hindi_selected
})

final_hindi_mb = current_size / (1024 * 1024)

print("\nFinal Hindi dataset selected")
print("-" * 60)
print("Records:", len(hindi_final))
print("Size:", round(final_hindi_mb, 2), "MB")


# ------------------------------------------------------------
# 13. Final statistics
# ------------------------------------------------------------

hindi_chars = hindi_final["text"].str.len()

print("\nFINAL HINDI STATISTICS")
print("=" * 60)

print("Total records:", len(hindi_final))
print("Total characters:", int(hindi_chars.sum()))
print("Average characters:", round(hindi_chars.mean(), 2))
print("Minimum characters:", int(hindi_chars.min()))
print("Maximum characters:", int(hindi_chars.max()))
print("Median characters:", float(hindi_chars.median()))
print("Final size:", round(final_hindi_mb, 2), "MB")


# ------------------------------------------------------------
# 14. Final validation
# ------------------------------------------------------------

print("\nFINAL VALIDATION")
print("=" * 60)

print("Missing values:")
print(hindi_final.isnull().sum())

print(
    "Empty records:",
    int((hindi_final["text"].str.strip() == "").sum())
)

print(
    "Duplicate records:",
    int(hindi_final["text"].duplicated().sum())
)


# ------------------------------------------------------------
# 15. Show sample records
# ------------------------------------------------------------

print("\nSAMPLE HINDI DATA")
print("=" * 60)

for i, text in enumerate(hindi_final["text"].head(5)):

    print(f"\n[{i}]")
    print(text[:1000])


# ------------------------------------------------------------
# 16. Export JSONL
# ------------------------------------------------------------

hindi_output = "bharat_hindi_clean.jsonl"

hindi_final.to_json(
    hindi_output,
    orient="records",
    lines=True,
    force_ascii=False
)

print("\nJSONL exported:")
print(hindi_output)


# ------------------------------------------------------------
# 17. Verify exported file
# ------------------------------------------------------------

hindi_file_size_mb = (
    os.path.getsize(hindi_output)
    / (1024 * 1024)
)

print("\nEXPORTED FILE")
print("=" * 60)
print("File:", hindi_output)
print("Size:", round(hindi_file_size_mb, 2), "MB")


# ------------------------------------------------------------
# 18. Reload JSONL and verify
# ------------------------------------------------------------

verified_hindi = pd.read_json(
    hindi_output,
    lines=True
)

print("\nRELOAD VERIFICATION")
print("=" * 60)
print("Rows:", len(verified_hindi))
print("Columns:", list(verified_hindi.columns))

print("\nMissing values:")
print(verified_hindi.isnull().sum())

print(
    "\nEmpty records:",
    int((verified_hindi["text"].str.strip() == "").sum())
)

print("\nFirst verified record:")
print(verified_hindi.iloc[0]["text"][:500])


# ------------------------------------------------------------
# 19. Final report
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("🇮🇳 BHARAT HINDI DATA — FINAL REPORT")
print("=" * 70)

print("Source              : AI4Bharat IndicCorpV2")
print("Language             : Hindi")
print("Script               : Devanagari")
print("Raw target           :", TARGET_RAW_MB, "MB")
print("Final target         :", TARGET_FINAL_MB, "MB")
print("Final JSONL size     :", round(hindi_file_size_mb, 2), "MB")
print("Final records        :", len(verified_hindi))
print("Missing values      :", int(verified_hindi.isnull().sum().sum()))
print("Empty records       :", int((verified_hindi["text"].str.strip() == "").sum()))
print("Duplicate records   :", int(verified_hindi["text"].duplicated().sum()))

print("\nSTATUS: HINDI DATA PREPARATION COMPLETE ✅")

Hindi Bharat dataset preparation started
Raw collection target   : 200 MB
Final dataset target    : 150 MB
Minimum characters      : 50



0it [00:00, ?it/s]


Collection complete
------------------------------------------------------------
Rows seen: 556273
Rows collected: 278137
Raw size: 200.0 MB

Initial DataFrame:
Rows: 278137
Columns: ['text']

Cleaning Hindi text...
Cleaning completed

Empty-text removal
Before: 278137
After : 278137
Removed: 0

Minimum-length filtering
Before: 278137
After : 238717
Removed: 39420

Duplicate removal
Before: 238717
After : 238698
Removed: 19

Devanagari ratio statistics
count    238698.000000
mean          0.783850
std           0.035534
min           0.227273
25%           0.771689
50%           0.789474
75%           0.803922
max           1.000000
Name: devanagari_ratio, dtype: float64

Hindi language filtering
Before: 238698
After : 238693
Removed: 5

Cleaned Hindi dataset
------------------------------------------------------------
Records: 238693
Size: 196.74 MB

Final Hindi dataset selected
------------------------------------------------------------
Records: 182476
Size: 150.0 MB

FINAL HINDI S

In [29]:
from datasets import load_dataset

english_stream = load_dataset(
    "ai4bharat/samanantar",
    "hi",
    split="train",
    streaming=True
)

print("Samanantar English-Hindi streaming dataset loaded")

README.md:   0%|          | 0.00/11.4k [00:00<?, ?B/s]

Samanantar English-Hindi streaming dataset loaded


In [30]:
for i, row in enumerate(english_stream):
    print(f"Record {i}")
    print(row)
    print("-" * 80)

    if i >= 4:
        break

Record 0
{'idx': 0, 'src': "However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles", 'tgt': 'आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।'}
--------------------------------------------------------------------------------
Record 1
{'idx': 1, 'src': 'Whosoever desires the reward of the world, with Allah is the reward of the world and of the Everlasting Life. Allah is the Hearer, the Seer.', 'tgt': 'और जो शख्स (अपने आमाल का) बदला दुनिया ही में चाहता है तो ख़ुदा के पास दुनिया व आख़िरत दोनों का अज्र मौजूद है और ख़ुदा तो हर शख्स की सुनता और सबको देखता है'}
--------------------------------------------------------------------------------
Record 2
{'idx': 2, 'src': 'The value of insects in the biosphere is enormous because they outnumber all other living groups in measure of species richness.', 'tgt': 'जैव-मंड

In [31]:
# ============================================================
# 🇮🇳 ENGLISH — BHARAT DATA PREPARATION
# Samanantar English-Hindi
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata
import os
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

TARGET_RAW_MB = 200
TARGET_FINAL_MB = 150

TARGET_RAW_BYTES = TARGET_RAW_MB * 1024 * 1024
TARGET_FINAL_BYTES = TARGET_FINAL_MB * 1024 * 1024

MIN_CHARS = 50

print("English Bharat dataset preparation started")
print(f"Raw collection target : {TARGET_RAW_MB} MB")
print(f"Final dataset target  : {TARGET_FINAL_MB} MB")
print(f"Minimum characters    : {MIN_CHARS}")


# ------------------------------------------------------------
# 2. Cleaning functions
# ------------------------------------------------------------

def normalize_unicode(text):
    if not isinstance(text, str):
        return ""

    return unicodedata.normalize("NFKC", text)


def remove_urls(text):
    url_pattern = r'https?://\S+|www\.\S+'
    return re.sub(url_pattern, ' ', text)


def remove_control_characters(text):
    cleaned = []

    for char in text:
        category = unicodedata.category(char)

        if category.startswith("C"):
            if char in "\n\t":
                cleaned.append(char)
        else:
            cleaned.append(char)

    return "".join(cleaned)


def normalize_whitespace(text):
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n\n', text)

    return text.strip()


def clean_text(text):
    text = normalize_unicode(text)
    text = remove_urls(text)
    text = remove_control_characters(text)
    text = normalize_whitespace(text)

    return text


# ------------------------------------------------------------
# 3. English quality check
# ------------------------------------------------------------

def latin_ratio(text):
    if not text:
        return 0.0

    latin_count = sum(
        1
        for char in text
        if ('A' <= char <= 'Z') or
           ('a' <= char <= 'z')
    )

    return latin_count / len(text)


# ------------------------------------------------------------
# 4. Collect approximately 200 MB raw English
# ------------------------------------------------------------

english_records = []

total_raw_bytes = 0
rows_seen = 0
rows_collected = 0

print("\nCollecting English data...")

for row in tqdm(english_stream):

    rows_seen += 1

    # English sentence is in the Samanantar 'src' field
    text = row.get("src", "")

    if not isinstance(text, str):
        continue

    text = text.strip()

    if not text:
        continue

    size = len(text.encode("utf-8"))

    english_records.append(text)

    total_raw_bytes += size
    rows_collected += 1

    if total_raw_bytes >= TARGET_RAW_BYTES:
        break


print("\nCollection complete")
print("-" * 60)
print("Rows seen:", rows_seen)
print("Rows collected:", rows_collected)
print(
    "Raw size:",
    round(total_raw_bytes / (1024 * 1024), 2),
    "MB"
)


# ------------------------------------------------------------
# 5. Create DataFrame
# ------------------------------------------------------------

english_df = pd.DataFrame({
    "text": english_records
})

print("\nInitial DataFrame")
print("Rows:", len(english_df))
print("Columns:", list(english_df.columns))


# ------------------------------------------------------------
# 6. Clean text
# ------------------------------------------------------------

print("\nCleaning English text...")

english_df["clean_text"] = english_df["text"].apply(clean_text)

print("Cleaning completed")


# ------------------------------------------------------------
# 7. Remove empty records
# ------------------------------------------------------------

before_empty = len(english_df)

english_df = english_df[
    english_df["clean_text"].str.strip().str.len() > 0
].copy()

after_empty = len(english_df)

print("\nEmpty-text removal")
print("Before:", before_empty)
print("After :", after_empty)
print("Removed:", before_empty - after_empty)


# ------------------------------------------------------------
# 8. Minimum-length filtering
# ------------------------------------------------------------

before_length = len(english_df)

english_df = english_df[
    english_df["clean_text"].str.len() >= MIN_CHARS
].copy()

after_length = len(english_df)

print("\nMinimum-length filtering")
print("Before:", before_length)
print("After :", after_length)
print("Removed:", before_length - after_length)


# ------------------------------------------------------------
# 9. Duplicate removal
# ------------------------------------------------------------

before_duplicates = len(english_df)

english_df = english_df.drop_duplicates(
    subset=["clean_text"]
).reset_index(drop=True)

after_duplicates = len(english_df)

print("\nDuplicate removal")
print("Before:", before_duplicates)
print("After :", after_duplicates)
print("Removed:", before_duplicates - after_duplicates)


# ------------------------------------------------------------
# 10. English language quality check
# ------------------------------------------------------------

english_df["latin_ratio"] = english_df["clean_text"].apply(
    latin_ratio
)

print("\nLatin-character ratio statistics")
print(english_df["latin_ratio"].describe())


# ------------------------------------------------------------
# 11. Keep English-dominant records
# ------------------------------------------------------------

before_language = len(english_df)

english_df = english_df[
    english_df["latin_ratio"] >= 0.50
].copy()

english_df = english_df.reset_index(drop=True)

after_language = len(english_df)

print("\nEnglish language filtering")
print("Before:", before_language)
print("After :", after_language)
print("Removed:", before_language - after_language)


# ------------------------------------------------------------
# 12. Calculate cleaned size
# ------------------------------------------------------------

english_df["bytes"] = english_df["clean_text"].apply(
    lambda x: len(x.encode("utf-8"))
)

cleaned_size_bytes = english_df["bytes"].sum()

cleaned_size_mb = cleaned_size_bytes / (1024 * 1024)

print("\nCleaned English dataset")
print("-" * 60)
print("Records:", len(english_df))
print("Size:", round(cleaned_size_mb, 2), "MB")


# ------------------------------------------------------------
# 13. Select approximately 150 MB final dataset
# ------------------------------------------------------------

english_selected = []

current_size = 0

for text in english_df["clean_text"]:

    size = len(text.encode("utf-8"))

    if current_size + size > TARGET_FINAL_BYTES:
        break

    english_selected.append(text)
    current_size += size


english_final = pd.DataFrame({
    "text": english_selected
})

final_english_mb = current_size / (1024 * 1024)

print("\nFinal English dataset selected")
print("-" * 60)
print("Records:", len(english_final))
print("Size:", round(final_english_mb, 2), "MB")


# ------------------------------------------------------------
# 14. Final statistics
# ------------------------------------------------------------

english_chars = english_final["text"].str.len()

print("\nFINAL ENGLISH STATISTICS")
print("=" * 60)

print("Total records:", len(english_final))
print("Total characters:", int(english_chars.sum()))
print(
    "Average characters:",
    round(english_chars.mean(), 2)
)
print(
    "Minimum characters:",
    int(english_chars.min())
)
print(
    "Maximum characters:",
    int(english_chars.max())
)
print(
    "Median characters:",
    float(english_chars.median())
)
print(
    "Final size:",
    round(final_english_mb, 2),
    "MB"
)


# ------------------------------------------------------------
# 15. Final validation
# ------------------------------------------------------------

print("\nFINAL VALIDATION")
print("=" * 60)

print("Missing values:")
print(english_final.isnull().sum())

print(
    "Empty records:",
    int(
        (english_final["text"].str.strip() == "").sum()
    )
)

print(
    "Duplicate records:",
    int(
        english_final["text"].duplicated().sum()
    )
)


# ------------------------------------------------------------
# 16. Show sample data
# ------------------------------------------------------------

print("\nSAMPLE ENGLISH DATA")
print("=" * 60)

for i, text in enumerate(
    english_final["text"].head(5)
):

    print(f"\n[{i}]")
    print(text[:1000])


# ------------------------------------------------------------
# 17. Export JSONL
# ------------------------------------------------------------

english_output = "bharat_english_clean.jsonl"

english_final.to_json(
    english_output,
    orient="records",
    lines=True,
    force_ascii=False
)

print("\nJSONL exported:")
print(english_output)


# ------------------------------------------------------------
# 18. Verify exported file
# ------------------------------------------------------------

english_file_size_mb = (
    os.path.getsize(english_output)
    / (1024 * 1024)
)

print("\nEXPORTED FILE")
print("=" * 60)

print("File:", english_output)
print(
    "Size:",
    round(english_file_size_mb, 2),
    "MB"
)


# ------------------------------------------------------------
# 19. Reload JSONL
# ------------------------------------------------------------

verified_english = pd.read_json(
    english_output,
    lines=True
)

print("\nRELOAD VERIFICATION")
print("=" * 60)

print("Rows:", len(verified_english))
print(
    "Columns:",
    list(verified_english.columns)
)

print("\nMissing values:")
print(verified_english.isnull().sum())

print(
    "\nEmpty records:",
    int(
        (verified_english["text"].str.strip() == "").sum()
    )
)

print("\nFirst verified record:")
print(
    verified_english.iloc[0]["text"][:500]
)


# ------------------------------------------------------------
# 20. Final report
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("🇮🇳 BHARAT ENGLISH DATA — FINAL REPORT")
print("=" * 70)

print("Source             : AI4Bharat Samanantar")
print("Configuration      : English-Hindi (hi)")
print("English field      : src")
print("Raw target         :", TARGET_RAW_MB, "MB")
print("Final target       :", TARGET_FINAL_MB, "MB")
print(
    "Final JSONL size   :",
    round(english_file_size_mb, 2),
    "MB"
)
print(
    "Final records      :",
    len(verified_english)
)
print(
    "Missing values     :",
    int(
        verified_english.isnull().sum().sum()
    )
)
print(
    "Empty records      :",
    int(
        (verified_english["text"].str.strip() == "").sum()
    )
)
print(
    "Duplicate records  :",
    int(
        verified_english["text"].duplicated().sum()
    )
)

print("\nSTATUS: ENGLISH DATA PREPARATION COMPLETE ✅")

English Bharat dataset preparation started
Raw collection target : 200 MB
Final dataset target  : 150 MB
Minimum characters    : 50



0it [00:00, ?it/s]


Collection complete
------------------------------------------------------------
Rows seen: 2143196
Rows collected: 2143196
Raw size: 200.0 MB

Initial DataFrame
Rows: 2143196
Columns: ['text']

Cleaning English text...
Cleaning completed

Empty-text removal
Before: 2143196
After : 2143168
Removed: 28

Minimum-length filtering
Before: 2143168
After : 1526245
Removed: 616923

Duplicate removal
Before: 1526245
After : 1345917
Removed: 180328

Latin-character ratio statistics
count    1.345917e+06
mean     8.054059e-01
std      4.205241e-02
min      1.892744e-02
25%      7.894737e-01
50%      8.141593e-01
75%      8.316832e-01
max      9.824561e-01
Name: latin_ratio, dtype: float64

English language filtering
Before: 1345917
After : 1345075
Removed: 842

Cleaned English dataset
------------------------------------------------------------
Records: 1345075
Size: 164.09 MB

Final English dataset selected
------------------------------------------------------------
Records: 1231482
Size: 150

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [34]:
import shutil
import os

drive_folder = "/content/drive/MyDrive/Bharat_LLM_Data"

os.makedirs(drive_folder, exist_ok=True)

shutil.copy(
    "bharat_hindi_clean.jsonl",
    os.path.join(drive_folder, "bharat_hindi_clean.jsonl")
)

shutil.copy(
    "bharat_english_clean.jsonl",
    os.path.join(drive_folder, "bharat_english_clean.jsonl")
)

print("Both files copied to Google Drive successfully.")
print(drive_folder)

Both files copied to Google Drive successfully.
/content/drive/MyDrive/Bharat_LLM_Data
